# Test for hudi

In [1]:
from pyspark.sql import SparkSession

# ============================================================
# Paths
# ============================================================

RAW_PATH    = "s3a://rawload"
BRONZE_PATH = "s3a://bronzeload"
SILVER_PATH = "s3a://silverload"
GOLD_PATH   = "s3a://goldload"

RAW_FILE = (
    f"{RAW_PATH}/agesexbyethnicgroup/Data8277.csv"
)

HUDI_PATH = f"{BRONZE_PATH}/agesexbyethnicgroup"
DELTA_PATH = f"{SILVER_PATH}/agesexbyethnicgroup"


# ============================================================
# Spark
# ============================================================

spark = (
    SparkSession.builder
    .appName("Three Table Format Comparison")

    # ========================================================
    # Hudi
    # ========================================================

    .config(
        "spark.jars",
        r"C:\data\datatool\spark\jars"
        r"\hudi-spark4.0-bundle_2.13-1.2.0.jar"
    )

    # ========================================================
    # Delta + Iceberg + Hadoop AWS
    # ========================================================

    .config(
        "spark.jars.packages",
        "io.delta:delta-spark_2.13:4.0.0,"
        "org.apache.iceberg:iceberg-spark-runtime-4.0_2.13:1.10.0,"
        "org.apache.hadoop:hadoop-aws:3.4.1"
    )

    # ========================================================
    # Spark SQL Extensions
    # ========================================================

    .config(
        "spark.sql.extensions",
        ",".join([
            "org.apache.spark.sql.hudi.HoodieSparkSessionExtension",
            "io.delta.sql.DeltaSparkSessionExtension",
            "org.apache.iceberg.spark.extensions.IcebergSparkSessionExtensions"
        ])
    )

    # ========================================================
    # Delta Catalog
    # ========================================================

    .config(
        "spark.sql.catalog.spark_catalog",
        "org.apache.spark.sql.delta.catalog.DeltaCatalog"
    )

    # ========================================================
    # Hudi
    # ========================================================

    .config(
        "spark.serializer",
        "org.apache.spark.serializer.KryoSerializer"
    )

    .config(
        "spark.kryo.registrator",
        "org.apache.spark.HoodieSparkKryoRegistrar"
    )

    # ========================================================
# Iceberg REST Catalog
# ========================================================

.config(
    "spark.sql.catalog.ice",
    "org.apache.iceberg.spark.SparkCatalog"
)

.config(
    "spark.sql.catalog.ice.type",
    "rest"
)

.config(
    "spark.sql.catalog.ice.uri",
    "http://localhost:8181"
)

# Explicitly use Iceberg S3FileIO
.config(
    "spark.sql.catalog.ice.io-impl",
    "org.apache.iceberg.aws.s3.S3FileIO"
)

# ========================================================
# Iceberg / AWS client
# ========================================================

.config(
    "spark.sql.catalog.ice.client.region",
    "us-east-1"
)

# ========================================================
# Iceberg S3 / MinIO
# ========================================================

.config(
    "spark.sql.catalog.ice.s3.endpoint",
    "http://127.0.0.1:9000"
)

.config(
    "spark.sql.catalog.ice.s3.access-key-id",
    "minioadmin"
)

.config(
    "spark.sql.catalog.ice.s3.secret-access-key",
    "minioadmin"
)

.config(
    "spark.sql.catalog.ice.s3.path-style-access",
    "true"
)

.config(
    "spark.sql.catalog.ice.s3.region",
    "us-east-1"
)

    # ========================================================
    # Hadoop S3A / MinIO
    #
    # Used by:
    #   Raw
    #   Hudi
    #   Delta
    # ========================================================

    .config(
        "spark.hadoop.fs.s3a.endpoint",
        "http://127.0.0.1:9000"
    )

    .config(
        "spark.hadoop.fs.s3a.access.key",
        "minioadmin"
    )

    .config(
        "spark.hadoop.fs.s3a.secret.key",
        "minioadmin"
    )

    .config(
        "spark.hadoop.fs.s3a.path.style.access",
        "true"
    )

    .config(
        "spark.hadoop.fs.s3a.impl",
        "org.apache.hadoop.fs.s3a.S3AFileSystem"
    )

    .config(
        "spark.hadoop.fs.s3a.aws.credentials.provider",
        "org.apache.hadoop.fs.s3a.SimpleAWSCredentialsProvider"
    )

    .getOrCreate()
)


# ============================================================
# Verification
# ============================================================

print("Spark :", spark.version)

print(
    "Hadoop:",
    spark.sparkContext._jvm
        .org.apache.hadoop.util.VersionInfo.getVersion()
)

print("\nIceberg namespaces:")
spark.sql("SHOW NAMESPACES IN ice").show()

Spark : 4.0.1
Hadoop: 3.4.1

Iceberg namespaces:
+---------+
|namespace|
+---------+
|  diamond|
|     gold|
+---------+



In [2]:
print(
    "REST URI:",
    spark.conf.get("spark.sql.catalog.ice.uri", "NOT SET")
)

print(
    "S3 region:",
    spark.conf.get("spark.sql.catalog.ice.s3.region", "NOT SET")
)

print(
    "Client region:",
    spark.conf.get("spark.sql.catalog.ice.client.region", "NOT SET")
)

print(
    "S3 endpoint:",
    spark.conf.get("spark.sql.catalog.ice.s3.endpoint", "NOT SET")
)

REST URI: http://localhost:8181
S3 region: us-east-1
Client region: us-east-1
S3 endpoint: http://127.0.0.1:9000


In [3]:
spark.sql("SHOW NAMESPACES IN ice").show()

+---------+
|namespace|
+---------+
|  diamond|
|     gold|
+---------+



In [4]:
raw_test = (
    spark.read
    .option("header", True)
    .csv(RAW_FILE)
)

raw_test.show(2, truncate=False)

+----+---+------+---+----+-----+
|Year|Age|Ethnic|Sex|Area|count|
+----+---+------+---+----+-----+
|2018|000|1     |1  |01  |795  |
|2018|000|1     |1  |02  |5067 |
+----+---+------+---+----+-----+
only showing top 2 rows


In [10]:
hudi_test = (
    spark.read
    .format("hudi")
    .load(HUDI_PATH)
)

hudi_test.show(2, truncate=False)

+-------------------+----------------------+------------------------------------+----------------------+-------------------------------------------------------------------------+----+---+------+---+------+-----+------------------------------------+
|_hoodie_commit_time|_hoodie_commit_seqno  |_hoodie_record_key                  |_hoodie_partition_path|_hoodie_file_name                                                        |Year|Age|Ethnic|Sex|Area  |count|_record_id                          |
+-------------------+----------------------+------------------------------------+----------------------+-------------------------------------------------------------------------+----+---+------+---+------+-----+------------------------------------+
|20260816094156160  |20260816094156160_5_15|8a860951-0fb1-415a-a226-09e1a9693a45|                      |479132ed-bddf-4318-acb9-1821a9b26e36-0_5-25-111_20260816094156160.parquet|2018|01 |5     |2  |335600|0    |8a860951-0fb1-415a-a226-09e1a9693a45|
|202

In [11]:
delta_test = (
    spark.read
    .format("delta")
    .load(DELTA_PATH)
)

delta_test.show(2, truncate=False)

+----+---+------+---+------+-----+------------------------------------+
|Year|Age|Ethnic|Sex|Area  |count|_record_id                          |
+----+---+------+---+------+-----+------------------------------------+
|2013|010|77    |9  |203800|..C  |52637146-cfd5-47c8-acc7-53b2823f425c|
|2013|010|77    |9  |203900|42   |fb52a95d-8fa8-4c1a-abb5-f058c33243dd|
+----+---+------+---+------+-----+------------------------------------+
only showing top 2 rows


In [12]:
spark.sql("SHOW TABLES IN ice.gold").show()

+---------+--------------------+-----------+
|namespace|           tableName|isTemporary|
+---------+--------------------+-----------+
|     gold|             dim_age|      false|
|     gold|            dim_area|      false|
|     gold|          dim_ethnic|      false|
|     gold|             dim_sex|      false|
|     gold|            dim_year|      false|
|     gold|fact_agesexbyethn...|      false|
+---------+--------------------+-----------+



In [13]:
spark.table(
    "ice.gold.fact_agesexbyethnicgroup"
).show(2, truncate=False)

+----+---+------+---+------+-----+
|Year|Age|Ethnic|Sex|Area  |count|
+----+---+------+---+------+-----+
|2006|000|1     |1  |323900|12.0 |
|2006|000|1     |2  |323900|12.0 |
+----+---+------+---+------+-----+
only showing top 2 rows


In [14]:
from pyspark.sql import Row

test_data = [
    Row(id=1, name="Alpha"),
    Row(id=2, name="Beta"),
    Row(id=3, name="Gamma"),
    Row(id=4, name="Delta"),
    Row(id=5, name="Epsilon")
]

test_df = spark.createDataFrame(test_data)

test_df.show()

+---+-------+
| id|   name|
+---+-------+
|  1|  Alpha|
|  2|   Beta|
|  3|  Gamma|
|  4|  Delta|
|  5|Epsilon|
+---+-------+



In [15]:
test_df.writeTo(
    "ice.diamond.spark_test"
).using(
    "iceberg"
).create()

In [ ]:
hadoop_conf = spark.sparkContext._jsc.hadoopConfiguration()

path = spark._jvm.org.apache.hadoop.fs.Path(
    "s3a://goldload/agesexbyethnicgroup"
)

fs = path.getFileSystem(hadoop_conf)

print("FS:", fs.getClass().getName())

for status in fs.listStatus(path):
    print(status.getPath())

In [ ]:
hadoop_conf = spark.sparkContext._jsc.hadoopConfiguration()

hadoop_conf.set(
    "fs.s3a.aws.credentials.provider",
    "org.apache.hadoop.fs.s3a.SimpleAWSCredentialsProvider"
)

hadoop_conf.set(
    "fs.s3a.access.key",
    "minioadmin"
)

hadoop_conf.set(
    "fs.s3a.secret.key",
    "minioadmin"
)

hadoop_conf.set(
    "fs.s3a.endpoint",
    "http://127.0.0.1:9000"
)

hadoop_conf.set(
    "fs.s3a.path.style.access",
    "true"
)

In [ ]:
spark.sparkContext._jvm.org.apache.hadoop.fs.FileSystem.closeAll()

In [ ]:
path = spark._jvm.org.apache.hadoop.fs.Path(
    "s3a://goldload/agesexbyethnicgroup"
)

fs = path.getFileSystem(hadoop_conf)

print("FS:", fs.getClass().getName())

for status in fs.listStatus(path):
    print(status.getPath())

In [ ]:
path = spark._jvm.org.apache.hadoop.fs.Path(
    "s3a://goldload"
)

fs = path.getFileSystem(hadoop_conf)

for status in fs.listStatus(path):
    print(status.getPath())

In [ ]:
print("Endpoint:",
      spark.sparkContext._jsc.hadoopConfiguration()
          .get("fs.s3a.endpoint"))

print("Access key:",
      spark.sparkContext._jsc.hadoopConfiguration()
          .get("fs.s3a.access.key"))

print("Secret key:",
      spark.sparkContext._jsc.hadoopConfiguration()
          .get("fs.s3a.secret.key"))

print("Provider:",
      spark.sparkContext._jsc.hadoopConfiguration()
          .get("fs.s3a.aws.credentials.provider"))

In [ ]:
spark._jsc.hadoopConfiguration().get("fs.s3a.endpoint")

In [ ]:
path = spark._jvm.org.apache.hadoop.fs.Path(
    "s3a://goldload/agesexbyethnicgroup"
)

fs = path.getFileSystem(
    spark.sparkContext._jsc.hadoopConfiguration()
)

for status in fs.listStatus(path):
    print(status.getPath())

In [ ]:
spark.sql("SHOW TABLES IN ice.gold").show()

In [ ]:
spark.sql("""
DESCRIBE EXTENDED ice.agesexbyethnicgroup
""").show(truncate=False)

In [ ]:
spark.sql("""
DESCRIBE EXTENDED ice.gold.fact_agesexbyethnicgroup
""").show(truncate=False)

In [ ]:
spark.sql("""
SELECT *
FROM ice.gold.fact_agesexbyethnicgroup.metadata_log_entries
ORDER BY timestamp DESC
LIMIT 5
""").show(truncate=False)

In [ ]:
df=spark.sql("""
SELECT
    'dim_year' AS table_name,
    file
FROM ice.gold.dim_year.metadata_log_entries
UNION ALL
SELECT
    'dim_age' AS table_name,
    file
FROM ice.gold.dim_age.metadata_log_entries
UNION ALL
SELECT
    'dim_ethnic' AS table_name,
    file
FROM ice.gold.dim_ethnic.metadata_log_entries
UNION ALL
SELECT
    'dim_sex' AS table_name,
    file
FROM ice.gold.dim_sex.metadata_log_entries
UNION ALL
SELECT
    'dim_area' AS table_name,
    file
FROM ice.gold.dim_area.metadata_log_entries
""")

df.writeTo("ice.diamond.anotherTable") \
  .using("iceberg") \
  .create()

In [ ]:
spark.table("ice.diamond.a_newTable").show()

In [ ]:
spark.stop()